# Vertex AI エージェント エンジンへの ADK エージェントのデプロイ

In [1]:
import importlib
import os
import warnings
import vertexai
from vertexai.preview import reasoning_engines
from vertexai.preview.reasoning_engines import AdkApp
import json
import uuid

from adk_agents import agents 
importlib.reload(agents)

print(f"Loaded Agent: {agents.root_agent.name}")

# 設定と環境変数の準備 [cite: 12, 15, 17]
warnings.filterwarnings("ignore")
LOCATION = "us-central1"
# PROJECT_ID = "qwiklabs-asl-00-8534302ca246" # 指定のプロジェクトID
PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
STAGING_BUCKET = f"gs://agent-deployment-{PROJECT_ID}-bucket" # [cite: 25, 26]

!gsutil ls $STAGING_BUCKET || gsutil mb -l $LOCATION $STAGING_BUCKET

client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

# Vertex AI SDKの初期化 
vertexai.init(project=PROJECT_ID, location=LOCATION)

# 1. エージェント定義の読み込み [cite: 31, 32]
# adk_agentsフォルダ内に、先ほど作成した agents.py と tools.py がある前提です
from adk_agents import agents
importlib.reload(agents)

# 2. ADKアプリケーションとしてラップ 
# root_agent（司令塔）をメインのエントリポイントにします
adk_app = AdkApp(agent=agents.root_agent)

# 3. Agent Engineへのデプロイ [cite: 34, 41]
DISPLAY_NAME = agents.root_agent.name
env_vars = {
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "true", # [cite: 39]
    "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "true", # [cite: 40]
}

print(f"Deploying agent: {DISPLAY_NAME}...")

# 一度作ったエージェントIDの呼び出し。
# UPDATEするときに必要。createするときには不要（割り当ててくれる）
ENGINE_ID = "projects/998865242242/locations/us-central1/reasoningEngines/6185745765239881728"
remote_app = client.agent_engines.get(name=ENGINE_ID)

#エージェント生成
# remote_app = client.agent_engines.create(
#     agent=adk_app, # [cite: 42, 59]
#     config={
#         "display_name": DISPLAY_NAME, # [cite: 44]
#         "description": "Linear vs NW Model Router Agent", # [cite: 45]
#         "requirements": [
#             "google-adk==1.22.1", # [cite: 48]
#             "google-genai==1.57.0", # [cite: 49]
#             "google-cloud-aiplatform[agent_engines,adk]", # [cite: 51]
#             "cloudpickle",
#             "pydantic",
#             "scikit-learn==1.2.2",
#             "pandas"
#             ],
#         "extra_packages": ["./adk_agents/"], # [cite: 52]
#         "env_vars": env_vars, # [cite: 53]
#         "staging_bucket": STAGING_BUCKET, # [cite: 54]
#     },
# )

# UPDATE
remote_app = client.agent_engines.update(
    name=remote_app.api_resource.name,
    agent=adk_app,
    config={
        "display_name": DISPLAY_NAME,
        "description": "Linear vs NW Model Router Agent　v3",
        "requirements": [
            "google-adk==1.22.1",
            "google-genai==1.57.0",
            "google-cloud-aiplatform[agent_engines,adk]",
            "cloudpickle",
            "pydantic",
            "scikit-learn==1.2.2",
            "pandas"
        ],
        "extra_packages": ["./adk_agents/"],
        "env_vars": env_vars,
        "staging_bucket": STAGING_BUCKET,
    },
)

Loaded Agent: prediction_coordinator
gs://agent-deployment-qwiklabs-asl-00-8534302ca246-bucket/agent_engine/
Deploying agent: prediction_coordinator...


In [2]:
def print_adk_output(event: dict):
    # content や parts が無いイベント（ステータス変更など）をスキップ
    content = event.get("content", {})
    parts = content.get("parts", [])
    if not parts:
        # contentが無い場合でも、ツール実行結果が直接 event に入っている場合がある
        if "function_response" in event:
             print("--- [Event: Tool execution finished] ---")
             return
        return

    part = parts[0]
    author = event.get("author", "unknown_agent")
    print(f"\n--- [Event: {author}] ---")

    if "function_call" in part:
        call = part["function_call"]
        print(f"Tool Call: {call.get('name')} (Args: {call.get('args')})")

    elif "function_response" in part:
        # ここ！ADKのバージョンによって階層が深い場合があります
        resp = part["function_response"]
        print(f"Tool Response from {resp.get('name')}:")
        print(f"  -> Result: {resp.get('response')}")

    elif "text" in part:
        print(f"Final Agent Response: {part['text'].strip()}")
    
    else:
        # 知らない形式が来たら、辞書の中身を全部見せる
        print(f"Raw Part Data: {json.dumps(part, indent=2, ensure_ascii=False)}")
    print(f"DEBUG: {part}")

In [3]:
USER_ID = "grad_test_user"
# remote_session = remote_app.create_session(user_id=USER_ID)
remote_session = await remote_app.async_create_session(user_id=USER_ID)

In [4]:
# 1. データの準備
data1 = {'X1': [0.62], 'X2': [808.5], 'X3': [367.5], 'X4': [220.5], 'X5': [3.5], 'X6': [2], 'X7': [0.4], 'X8': [3]} # 16.47

input_text = f"線形モデルを使用して、以下のデータの予測結果を出してください。データ: {data1}"

# 2. engine.resource_name を使って直接クエリを投げる
try:
    async for event in remote_app.async_stream_query(
            user_id=USER_ID,
            session_id=remote_session["id"],
            message=input_text,
        ):
        print_adk_output(event)
    # # ADKの場合、レスポンスが辞書で返ってくることが多いので中身を確認
    # print(f"Final Response: {response}")
except Exception as e:
    print(f"Execution Error: {e}")
    # もしこれでもダメな場合、最終手段として以下の .set_up() を試す必要があります


--- [Event: prediction_coordinator] ---
Tool Call: predict_linear_model (Args: {'input_data': {'X3': [367.5], 'X5': [3.5], 'X6': [2], 'X4': [220.5], 'X2': [808.5], 'X7': [0.4], 'X1': [0.62], 'X8': [3]}})
DEBUG: {'function_call': {'id': 'adk-83dbb8ed-4107-40bc-a820-7a365c042fa7', 'args': {'input_data': {'X3': [367.5], 'X5': [3.5], 'X6': [2], 'X4': [220.5], 'X2': [808.5], 'X7': [0.4], 'X1': [0.62], 'X8': [3]}}, 'name': 'predict_linear_model'}, 'thought_signature': 'CocFAY89a18UYunxYJEMFh21GmeMPuxtBxL9Wvd0qs9wGPZxT6CmQO5AhNt0rVz4twLwVCdvccvyyPjxSUvUjD8DoAJTGq_qkCyORtrIcFVrF_yOVXt320ExwUGKm4ewQCW4hbNlZFRgiJ_fSx3FBIlbFDaVBzl184dKhOtRl5XMr6y-OOmoTwLN3ipLKP-ZwJy9qMfD2d8ZvAstMApIy-yBX6Uw42okruOIZHVkbSQIf-yh6Om4NaZ0xebcwbBoWTCeR0gSaNNJDHSfLTOlURDy2MyZp8xoNAISTBlw_tjrpn8oQl_vx0kyqaaQbqtUzNyn_QxVnvlIdShX3vYlkk9yGqK-eptH6MQ2fl42MJ7ChWU7UY8tzvmUj17mF1J3vEigWm4xbzM4v00NkuxMlsQI-7Zln2JZoFnu88TpxVHkzfEg_eXdw029ldSUbXF3-tJe97WQXXd8TZ4BoLFbXhHkrGtYO2_GGX6M2m1f8fKSYiQfR5sqtL8J0a0Tyh4KZZmlTE-KK0_nrt-L_I6

In [5]:

# テスト2: NWモデルの呼び出し
print("\n--- Testing NW Model ---")

# テスト用のクエリ（NWモデルを明示的に指定してルーターの挙動も確認します）
input_text = f"高度な非線形計算を行うニューラルネットワークモデルを使用して、以下のデータの予測結果を出してください。データ: {data1}"

try:
    async for event in remote_app.async_stream_query(
            user_id=USER_ID,
            session_id=remote_session["id"],
            message=input_text,
        ):
        print_adk_output(event)
    # # ADKの場合、レスポンスが辞書で返ってくることが多いので中身を確認
    # print(f"Final Response: {response}")
except Exception as e:
    print(f"Execution Error: {e}")
    # もしこれでもダメな場合、最終手段として以下の .set_up() を試す必要があります


--- Testing NW Model ---

--- [Event: prediction_coordinator] ---
Tool Call: predict_nw_model (Args: {'input_data': {'X6': [2], 'X1': [0.62], 'X5': [3.5], 'X3': [367.5], 'X8': [3], 'X4': [220.5], 'X7': [0.4], 'X2': [808.5]}})
DEBUG: {'function_call': {'id': 'adk-5228e36d-487d-4c56-b547-a46f3e088fb5', 'args': {'input_data': {'X6': [2], 'X1': [0.62], 'X5': [3.5], 'X3': [367.5], 'X8': [3], 'X4': [220.5], 'X7': [0.4], 'X2': [808.5]}}, 'name': 'predict_nw_model'}, 'thought_signature': 'Cr0DAY89a19j0hp0-0cVjKnHfJPf1CANx1fFyNqp8nL1g5XrBJlIBFZkA440BBVAwdMvc4De8Z2IcNgWbMKQfyfFJ_wagrnc3ESZF9t5ow5NRpMpGHFiyDq__MNnuzrohOG5lOWD1kNm9GlsQhYHAAOsfixmppNzR1QBI0pIf28x-eb8kXFjC8PPg6YjgzsWI2Ia0PxDNN2qo81kxwIJuHQi8_jSpe0EbT1Py69YL7q39YcoNXE1gFzU5xjYQe8-j8PSaNQ1cEMySlsU-Yt7z5MtAv_o89wmLgN92V-XpHuE7P5YELcMqyI8RwCN6sMdUs1uK_N_OrR98CZODK2OFL35OiRVtNhN_jD165SCBd083zCTErv75uuL9c0VBS0zS7sdZ_f_YcTENO5GqzTTiy52PPyk-3AGUC-3M02igTXJ23lfLVnz57IUpm1NN5nuLQVmZXR8VwjjDBuNtHqV9ttGq3cM5v_Ih7H4cSuam8kkuFI7SlNW2P5C8NjHzd_VG

In [6]:
# テスト3: ガウスモデルの呼び出し
print("\n--- Testing Gauss Model ---")

# テスト用のクエリ（NWモデルを明示的に指定してルーターの挙動も確認します）
input_text = f"ガウス推定モデルを使用して、以下のデータの予測結果を出してください。データ: {data1}"

try:
    async for event in remote_app.async_stream_query(
            user_id=USER_ID,
            session_id=remote_session["id"],
            message=input_text,
        ):
        print_adk_output(event)
    # # ADKの場合、レスポンスが辞書で返ってくることが多いので中身を確認
    # print(f"Final Response: {response}")
except Exception as e:
    print(f"Execution Error: {e}")
    # もしこれでもダメな場合、最終手段として以下の .set_up() を試す必要があります


--- Testing Gauss Model ---

--- [Event: prediction_coordinator] ---
Tool Call: predict_Gauss_model (Args: {'input_data': {'X7': [0.4], 'X1': [0.62], 'X2': [808.5], 'X8': [3], 'X6': [2], 'X4': [220.5], 'X5': [3.5], 'X3': [367.5]}})
DEBUG: {'function_call': {'id': 'adk-4a5933e7-1e52-4305-9233-de9793ec62a2', 'args': {'input_data': {'X7': [0.4], 'X1': [0.62], 'X2': [808.5], 'X8': [3], 'X6': [2], 'X4': [220.5], 'X5': [3.5], 'X3': [367.5]}}, 'name': 'predict_Gauss_model'}, 'thought_signature': 'CvgCAY89a18rmREv5D_9Xr4xR24DsS4-umnb_hL7Er0vx01MW_-I2VHV0LdXkbK3FTxIPr8c477mAGE_Go4I3Rm556bC5wMCllZl10gY5cwtA20FSKka4P4p-y-kvggyaub-UigEGeEOG8DNekL7drJ5Emcz738sDEsE1_P7XbUHsgDujklFrVWEpyAgjCmKKwGg3tr8bSsxamWkp6TtRrEcN-a1f5iB04VZxT2vnPig3JnOHpA0UEh_Nar7YaTrc4lLm6D5R2yhsC4uQVNU0hbRqtTiVVkVGqw1KsBuy22VTLfthvraN9Lavs8SdCf2j-9fG61sOvuHjqq_Y2zGRds-W3zvth5mZrRx6QrP5d58o5_mGVMK4GvCcfNE2mvhMz6O-fEVOGIeKM2MbqkG15rAFeKAmpWiCwINlf0nbmi37l1IFopj1tNYGXVYibFK-Geob2zh4suSyR_ZUTbyXssT9ZLPbnsm84naSBmPNST2t1vAlyS5CH-p

In [7]:
%%bash
# いつものお掃除
fuser -k 8000/tcp || true
sleep 2

# パスの設定
PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
echo "--------------------------------------------------------"
echo "👇 👇 👇 起動したらこちらをクリック 👇 👇 👇"
echo "🔗 https://${PROXY_BASE}/proxy/8000/dev-ui/"
echo "--------------------------------------------------------"

# 魔法の1行：Pythonに現在のフォルダの位置を教える
export PYTHONPATH=$(pwd)

# 起動（ディレクトリを「.」＝現在のフォルダ全体 にします）
adk web . --url_prefix /proxy/8000
adk web adk_agents --url_prefix /proxy/8000

--------------------------------------------------------
👇 👇 👇 起動したらこちらをクリック 👇 👇 👇
🔗 https://4675fc93ba3b161b-dot-us-central1.notebooks.googleusercontent.com/proxy/8000/dev-ui/
--------------------------------------------------------


/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
2026-03-13 08:11:49,535 - INFO - service_factory.py:220 - Using in-memory memory service
2026-03-13 08:11:49,536 - INFO - local_storage.py:83 - Using per-agent session storage rooted at /home/jupyter
2026-03-13 08:11:49,536 - INFO - local_storage.py:109 - Using file artifact service at /home/jupyter/.adk/artifacts
/opt/conda/lib/python3.10/site-packages/google/adk/cli/fast_api.py:141: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking cha


+-----------------------------------------------------------------------------+
| ADK Web Server started                                                      |
|                                                                             |
| For local testing, access at http://127.0.0.1:8000.                         |
+-----------------------------------------------------------------------------+



INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:56594 - "GET /dev-ui/assets/config/runtime-config.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:56600 - "GET /list-apps?relative_path=./ HTTP/1.1" 200 OK


...

INFO:     Shutting down
INFO:     Waiting for application shutdown.



+-----------------------------------------------------------------------------+
| ADK Web Server shutting down...                                             |
+-----------------------------------------------------------------------------+



2026-03-13 08:14:58,552 - INFO - runners.py:1479 - Closing runner...
2026-03-13 08:14:58,552 - INFO - runners.py:1487 - Runner closed.
INFO:     Application shutdown complete.
INFO:     Finished server process [25791]

Aborted!


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType